# Daily Practice — 2026-10-31 — ML Testing & MLOps: Building a Regression Test Suite for Model Retraining

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) —
569 real patient samples with 30 real-valued tumor measurements (radius, texture, perimeter, area,
smoothness, etc.) computed from digitized images of fine needle aspirate biopsies, with a binary
malignant/benign label. Loaded via `sklearn.datasets.load_breast_cancer`.

**Problem statement:** In production ML systems, every retrain is a risk: a new model could silently
perform worse than the one it replaces, disagree with it on many individual predictions, or fail to
reproduce consistently across runs even with a fixed random seed. You are the QA engineer on a
diagnostics team. Before any retrained model is allowed to replace the current production model, a
CI pipeline needs an automated regression test suite to catch these failure modes. This notebook sets
up a "baseline" (production) model and a "candidate" (retrained) model on the same real dataset — your
job is to write the tests that decide whether the candidate is safe to ship.

**What you should produce:** four plain Python test functions (pytest-style, but runnable without
pytest — each returns `(passed: bool, message: str)`):

1. `test_performance_regression(...)` — fails if the candidate's recall, precision, or ROC-AUC drops
   more than a tolerance below the baseline's recorded metrics. Recall should get the tightest
   tolerance: for a cancer diagnostic, a missed malignant case (false negative) is far more costly
   than a false positive, so a regression there matters more than an equal-sized drop in precision.
2. `test_prediction_drift(...)` — fails if the candidate disagrees with the baseline on more than some
   percentage of individual held-out predictions, even if the aggregate metrics look fine.
3. `test_reproducibility(...)` — fits the same model architecture with the same fixed random seed `N`
   times and fails if any run's recall differs from the others beyond a tiny floating-point tolerance
   (this catches hidden, unintended sources of nondeterminism in a training pipeline).
4. `test_input_schema(...)` — fails if an incoming feature table doesn't match the column names/order
   the model was trained on, or if any column's values fall outside an expected range.

Then run all four tests against the baseline and candidate models built below, and print a clear
`PASS`/`FAIL` report for each, with a short message explaining the result.


## Setup & data

Load the real dataset and split off a held-out set that neither model has seen. This held-out set
plays the role of "fresh production traffic" that both the baseline and the candidate get scored on.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
X = data.frame.drop(columns=["target"])
y = data.frame["target"]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("train shape:", X_train.shape, "holdout shape:", X_holdout.shape)
print("class balance (train):", y_train.value_counts(normalize=True).round(3).to_dict())


## Part 1 — the "production" baseline model

This is the model currently serving traffic. Its metrics on the holdout set get recorded once, the
way a real monitoring system would snapshot a deployed model's offline evaluation metrics.


In [ ]:
baseline_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_holdout)
baseline_proba = baseline_model.predict_proba(X_holdout)[:, 1]

baseline_metrics = {
    "precision": precision_score(y_holdout, baseline_preds),
    "recall": recall_score(y_holdout, baseline_preds),
    "roc_auc": roc_auc_score(y_holdout, baseline_proba),
}
print("Recorded production metrics:", baseline_metrics)


## Part 2 — a "retrained" candidate model

Someone on the team retuned the hyperparameters (fewer, shallower trees) and wants to ship this as
the new production model. This is exactly the kind of change that should go through your test suite
before deployment.


In [ ]:
candidate_model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE)
candidate_model.fit(X_train, y_train)

candidate_preds = candidate_model.predict(X_holdout)
candidate_proba = candidate_model.predict_proba(X_holdout)[:, 1]

candidate_metrics = {
    "precision": precision_score(y_holdout, candidate_preds),
    "recall": recall_score(y_holdout, candidate_preds),
    "roc_auc": roc_auc_score(y_holdout, candidate_proba),
}
print("Candidate metrics:", candidate_metrics)


## Part 3 — write the regression test suite (your task)

Fill in each function below. Each must return a `(passed, message)` tuple — do not just `print` and
return `None`, since a CI gate needs the boolean to decide pass/fail programmatically.


In [ ]:
def test_performance_regression(baseline_metrics, candidate_metrics, tolerance=0.02):
    """
    TODO: Return (passed: bool, message: str).
    Fail if candidate's recall, precision, or roc_auc drops more than `tolerance`
    below the corresponding baseline metric. Recall matters most for this
    diagnostic use case -- consider giving it a tighter tolerance than the rest.
    """
    # TODO: implement
    raise NotImplementedError


def test_prediction_drift(baseline_preds, candidate_preds, max_disagreement_rate=0.05):
    """
    TODO: Return (passed: bool, message: str).
    Fail if the fraction of holdout samples where baseline_preds and
    candidate_preds disagree exceeds `max_disagreement_rate`.
    """
    # TODO: implement
    raise NotImplementedError


def test_reproducibility(model_fn, X, y, random_state=RANDOM_STATE, n_runs=3, atol=1e-9):
    """
    TODO: Return (passed: bool, message: str).
    Call model_fn(random_state) n_runs times, fit+evaluate each run on the same
    X, y, and fail if the recall of any run differs from the others by more than
    `atol`. `model_fn` is a factory: model_fn(seed) -> unfitted sklearn estimator.
    """
    # TODO: implement
    raise NotImplementedError


def test_input_schema(X, expected_columns, expected_dtypes=None, value_ranges=None):
    """
    TODO: Return (passed: bool, message: str).
    Fail if X's columns don't exactly match `expected_columns` (names + order).
    If `expected_dtypes` is given (dict col -> dtype string), also check dtypes.
    If `value_ranges` is given (dict col -> (min, max)), also fail when any
    column's values fall outside that range.
    """
    # TODO: implement
    raise NotImplementedError


## Part 4 — run the suite and report results

Call each test against the baseline/candidate objects built above. Think carefully about what
`model_fn` should be for `test_reproducibility`: it needs to build a *fresh, unfitted* model with the
candidate's architecture each time -- should every run get the *same* seed, or a *different* one?
(Hint: the goal is to catch nondeterminism, not to test random variation.)


In [ ]:
# TODO: build the `results` dict by calling each test function, then print a
# PASS/FAIL report. For example (uncomment and complete):
#
# results = {
#     "performance_regression": test_performance_regression(baseline_metrics, candidate_metrics),
#     "prediction_drift": test_prediction_drift(baseline_preds, candidate_preds),
#     "reproducibility": test_reproducibility(
#         lambda seed: RandomForestClassifier(n_estimators=50, max_depth=3, random_state=seed),
#         X_train, y_train,
#     ),
#     "input_schema": test_input_schema(X_holdout, expected_columns=list(X_train.columns)),
# }
#
# for name, (passed, message) in results.items():
#     print(f"[{'PASS' if passed else 'FAIL'}] {name}: {message}")


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

**Key design decisions, explained:**

- `test_performance_regression` gives `recall` half the general tolerance, because in a cancer
  diagnostic a false negative (missed malignant case) is far more costly than a false positive. This
  is the evaluation-metric failure mode every ML QA engineer should watch for: a model can look fine
  on precision or accuracy while quietly getting worse at the metric that actually matters for the
  use case.
- `test_prediction_drift` compares individual predictions, not just aggregate metrics, because two
  models can have nearly identical recall/precision while flipping predictions on a *different subset*
  of patients. Aggregate metrics hide exactly which cases changed.
- `test_reproducibility` calls `model_fn(RANDOM_STATE)` with the **same** fixed seed on every run, on
  purpose — the point is not randomness, it's determinism. If a fixed seed doesn't produce identical
  results, something in the pipeline (a library version, thread-level nondeterminism, unstable data
  ordering) is silently nondeterministic, which is a real bug worth catching before it reaches
  production.
- `test_input_schema` checks column identity (not just column *count*) and, where given, numeric
  ranges. A silently reordered or out-of-range column is one of the most common real-world causes of
  a model failing in production despite passing every offline metric.

```python
def test_performance_regression(baseline_metrics, candidate_metrics, tolerance=0.02):
    recall_tolerance = tolerance / 2
    failures = []
    for metric, tol in [("precision", tolerance), ("recall", recall_tolerance), ("roc_auc", tolerance)]:
        drop = baseline_metrics[metric] - candidate_metrics[metric]
        if drop > tol:
            failures.append(f"{metric} dropped by {drop:.4f} (tolerance {tol:.4f})")
    passed = len(failures) == 0
    message = "no regression" if passed else "; ".join(failures)
    return passed, message


def test_prediction_drift(baseline_preds, candidate_preds, max_disagreement_rate=0.05):
    disagreement_rate = np.mean(np.asarray(baseline_preds) != np.asarray(candidate_preds))
    passed = disagreement_rate <= max_disagreement_rate
    message = f"disagreement rate {disagreement_rate:.4f} (max allowed {max_disagreement_rate:.4f})"
    return passed, message


def test_reproducibility(model_fn, X, y, random_state=RANDOM_STATE, n_runs=3, atol=1e-9):
    recalls = []
    for _ in range(n_runs):
        model = model_fn(random_state)
        model.fit(X, y)
        preds = model.predict(X)
        recalls.append(recall_score(y, preds))
    spread = max(recalls) - min(recalls)
    passed = spread <= atol
    message = f"recall spread across {n_runs} runs = {spread:.2e} (tolerance {atol:.1e}), recalls={recalls}"
    return passed, message


def test_input_schema(X, expected_columns, expected_dtypes=None, value_ranges=None):
    failures = []
    if list(X.columns) != list(expected_columns):
        failures.append("column names/order mismatch")
    if expected_dtypes:
        for col, dtype in expected_dtypes.items():
            if str(X[col].dtype) != dtype:
                failures.append(f"{col} dtype {X[col].dtype} != expected {dtype}")
    if value_ranges:
        for col, (lo, hi) in value_ranges.items():
            if X[col].min() < lo or X[col].max() > hi:
                failures.append(f"{col} out of range [{lo}, {hi}]")
    passed = len(failures) == 0
    message = "schema OK" if passed else "; ".join(failures)
    return passed, message


results = {
    "performance_regression": test_performance_regression(baseline_metrics, candidate_metrics),
    "prediction_drift": test_prediction_drift(baseline_preds, candidate_preds),
    "reproducibility": test_reproducibility(
        lambda seed: RandomForestClassifier(n_estimators=50, max_depth=3, random_state=seed),
        X_train, y_train,
    ),
    "input_schema": test_input_schema(X_holdout, expected_columns=list(X_train.columns)),
}

for name, (passed, message) in results.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {message}")
```

With `tolerance=0.02`, the candidate (fewer, shallower trees) typically **fails**
`test_performance_regression` on recall -- exactly the kind of regression a CI gate should block
before it reaches patients, even though the model still "looks reasonable" overall.
`test_reproducibility` passes because `RandomForestClassifier` with a fixed seed is deterministic on
a single machine; try dropping `random_state` from `model_fn` to see the test correctly fail instead.

</details>
